In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [2]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

We still load the embedding model to encode the query, but we don't re-embed all the documents. No fit call needed, because the index is already built and waiting on disk.

This is the same two-process split we used for text search in module 1. One process ingests and builds the index, another queries it.

It matters more here than with text search. Embedding the whole dataset takes about a minute. We don't want a user waiting that long when the app starts up. We pay that cost once during ingestion, and the query side starts up instantly.

In [3]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [4]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. You can start learning and submitting homework while the form is open, even if the program has already begun.'

In [5]:
vs_index.close()